# Aspect Based Sentiment Analysis Pipeline

In [2]:
import pandas as pd
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [4]:
df=pd.read_csv('nifty50_tweets.csv')

In [7]:
df.head()

,timestamp,ticker,tweet,user_id,username,followers_count,verified,user_location,user_created_at,clauses
0,2025-09-20 23:24:18,BHARTIARTL,"Debt levels rising, investors cautious affecti...",48,@User0048,265196,False,"Hyderabad, India",2020-05-19,"[Debt levels rising, investors cautious affect..."
1,2025-09-14 07:44:42,HDFC,"Debt levels rising, investors cautious affecti...",230,@User0230,122948,False,"Mumbai, India",2023-06-29,"[Debt levels rising, investors cautious affect..."
2,2025-09-12 09:36:07,INFY,"🚀 INFY: Leadership change announced, managemen...",408,@User0408,228039,False,"Delhi, India",2016-03-04,"[🚀 INFY:, Leadership change announced, managem..."
3,2025-09-11 02:40:13,HDFCAMC,HDFCAMC: Regulatory scrutiny looming over company,359,@User0359,354424,False,"Pune, India",2010-05-30,[HDFCAMC: Regulatory scrutiny looming over com...
4,2025-09-08 00:58:23,DRREDDY,"Dividend announced, fundamentals solid affecti...",129,@User0129,337565,False,"Kolkata, India",2019-01-25,"[Dividend announced, fundamentals solid affect..."


## Extracting Clause using Spacy

In [5]:
nlp = spacy.load("en_core_web_sm")

def split_clauses(tweet):
    doc = nlp(tweet)
    return [sent.text.strip() for sent in doc.sents]

df['clauses'] = df['tweet'].apply(split_clauses)

# Standard Aspects and the embedings 

In [ ]:
model_embed = SentenceTransformer("all-MiniLM-L6-v2")

aspect_desc = {
    "management": "company leadership, CEO, executives, board members, management decisions",
    "governance": "government policy, regulations, subsidies, lawsuits, political decisions",
    "fundamentals": "financial performance, revenue, profit, earnings, debt, cash flow",
    "hype": "market hype, excitement, , FOMO, speculation, rumors",
    "other": "miscellaneous, unknown, not covered, unrelated topics"  
}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
aspect_embeddings = {a: model_embed.encode(desc, normalize_embeddings=True) for a, desc in aspect_desc.items()}

def soft_aspect_assign(clause, tau=0.5):
    emb = model_embed.encode(clause, normalize_embeddings=True)
    sims = {a: util.cos_sim(emb, aspect_embeddings[a]).item() for a in aspect_embeddings}
    vals = np.array(list(sims.values()))
    exp = np.exp(vals / tau)
    probs = exp / exp.sum()
    return dict(zip(sims.keys(), probs))

## Sentiment Classification

In [10]:
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model_sent = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

def get_sentiment(clause):
    inputs = tokenizer(clause, return_tensors="pt", truncation=True)
    outputs = model_sent(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    labels = ["positive", "neutral", "negative"]
    return labels[probs.argmax()]

config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

In [11]:
def absa(clause):
    aspects = soft_aspect_assign(clause)
    sentiment = get_sentiment(clause)
    sentiment_map = {"positive":1, "neutral":0, "negative":-1}
    score = sentiment_map[sentiment]
    return {a: score*prob for a, prob in aspects.items()}

In [12]:
all_aspects = ["management","governance","fundamentals","hype","other"]

def absa_per_tweet(clauses):
    absa_scores = [absa(clause) for clause in clauses]
    avg_scores = {a: np.mean([score[a] for score in absa_scores]) for a in all_aspects}
    return avg_scores

df['absa'] = df['clauses'].apply(absa_per_tweet)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [13]:
df.head()

,timestamp,ticker,tweet,user_id,username,followers_count,verified,user_location,user_created_at,clauses,absa
0,2025-09-20 23:24:18,BHARTIARTL,"Debt levels rising, investors cautious affecti...",48,@User0048,265196,False,"Hyderabad, India",2020-05-19,"[Debt levels rising, investors cautious affect...","{'management': -0.18826897470473103, 'governan..."
1,2025-09-14 07:44:42,HDFC,"Debt levels rising, investors cautious affecti...",230,@User0230,122948,False,"Mumbai, India",2023-06-29,"[Debt levels rising, investors cautious affect...","{'management': -0.19761728755776256, 'governan..."
2,2025-09-12 09:36:07,INFY,"🚀 INFY: Leadership change announced, managemen...",408,@User0408,228039,False,"Delhi, India",2016-03-04,"[🚀 INFY:, Leadership change announced, managem...","{'management': 0.27127892983153346, 'governanc..."
3,2025-09-11 02:40:13,HDFCAMC,HDFCAMC: Regulatory scrutiny looming over company,359,@User0359,354424,False,"Pune, India",2010-05-30,[HDFCAMC: Regulatory scrutiny looming over com...,"{'management': -0.20769072619925188, 'governan..."
4,2025-09-08 00:58:23,DRREDDY,"Dividend announced, fundamentals solid affecti...",129,@User0129,337565,False,"Kolkata, India",2019-01-25,"[Dividend announced, fundamentals solid affect...","{'management': 0.0, 'governance': 0.0, 'fundam..."


## Aggregating for further Use

In [14]:
absa_df = pd.concat([df.drop(['absa','clauses'], axis=1), df['absa'].apply(pd.Series)], axis=1)

In [15]:
absa_df['date'] = pd.to_datetime(absa_df['timestamp']).dt.date
daily_absa = absa_df.groupby(['ticker','date'])[all_aspects].mean().reset_index()

In [16]:
daily_absa.to_csv("daily_absa_vectors_with_other.csv", index=False)
print(daily_absa.head())

       ticker        date  management  governance  fundamentals      hype  \
0  ADANIGREEN  2025-09-01   -0.275925   -0.148510     -0.200190 -0.259016   
1  ADANIGREEN  2025-09-02   -0.165524   -0.169660     -0.240057 -0.278622   
2  ADANIGREEN  2025-09-03    0.258100    0.167915      0.190997  0.222662   
3  ADANIGREEN  2025-09-07   -0.046948    0.009291     -0.023721  0.043236   
4  ADANIGREEN  2025-09-10    0.000000    0.000000      0.000000  0.000000   

      other  
0 -0.116358  
1 -0.146137  
2  0.160326  
3  0.018142  
4  0.000000  
